In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_processor import TennisDataProcessor
from src.feature_engineer import TennisFeatureEngineer

In [ ]:
print("=== COMPLETE TENNIS PREDICTION PIPELINE TEST ===")

processor = TennisDataProcessor()
matches = processor.load_matches_data(data_path='../data/raw/')
clean_matches = processor.clean_matches_data()

print(f"Data loaded: {len(processor.matches_df)} matches")
print(f"Date range: {processor.matches_df['tourney_date'].min()} to {processor.matches_df['tourney_date'].max()}")

# Initialize feature engineer
feature_engineer = TennisFeatureEngineer(processor)
print("Feature engineer initialized")

In [ ]:
print("=== SINGLE MATCH FEATURE GENERATION TEST ===")

# Test with a specific match from the dataset
test_match = processor.matches_df.iloc[10305]  # Pick a match in the middle
sorted_players = sorted([test_match['winner_name'], test_match['loser_name']])
player1, player2 = sorted_players[0], sorted_players[1]

print(f"Test match: {player1} vs {player2}")
print(f"Winner name: {test_match['winner_name']} ({'player1' if player1 == test_match['winner_name'] else 'player2'})")
print(f"Loser name: {test_match['loser_name']} ({'player1' if player1 == test_match['loser_name'] else 'player2'})")
print(f"Date: {test_match['tourney_date']}")
print(f"Surface: {test_match['surface']}")
print(f"Tournament: {test_match['tourney_name']}")

# Generate features for this match
features = feature_engineer.create_match_features(
    test_match['tourney_date'],
    test_match['winner_name'],
    test_match['loser_name'],
    test_match['surface']
)

print(f"\nGenerated {len(features)} features:")
for feature_name, value in features.items():
    print(f"  {feature_name}: {value:.4f}" if isinstance(value, float) else f"  {feature_name}: {value}")

# Sanity checks
print(f"\n=== FEATURE SANITY CHECKS ===")
print(f"ELO ratings reasonable (1200-2200): {1200 <= features['player1_elo'] <= 2200 and 1200 <= features['player2_elo'] <= 2200}")
print(f"Form values in [0,1]: {0 <= features['player1_recent_form'] <= 1 and 0 <= features['player2_recent_form'] <= 1}")
print(f"Surface encoded correctly: {features['surface_hard'] + features['surface_clay'] + features['surface_grass'] == 1}")
print(f"H2H advantages centered around 0: {-0.5 <= features['h2h_overall_advantage'] <= 0.5}")

In [ ]:
print("=== TRAINING DATASET CREATION TEST ===")

# Create a small training dataset for testing (recent matches only)
recent_cutoff = processor.matches_df['tourney_date'].max() - pd.Timedelta(days=365)
print(f"Creating training dataset for matches after {recent_cutoff}")

training_df = feature_engineer.create_training_dataset(
    start_date=recent_cutoff,
    min_matches_for_inclusion=10
)

print(f"\nTraining dataset shape: {training_df.shape}")
print(f"Feature columns: {len([col for col in training_df.columns if col not in ['match_date', 'player1', 'player2', 'surface', 'tourney_level', 'target']])}")

# Validate the dataset
print(f"\n=== TRAINING DATASET VALIDATION ===")
print(f"Target balance:")
print(training_df['target'].value_counts())
print(f"Balance ratio: {training_df['target'].mean():.3f}")

# Check for missing values
missing_values = training_df.isnull().sum()
critical_missing = missing_values[missing_values > 0]
if len(critical_missing) > 0:
    print(f"FAIL: Found missing values:")
    print(critical_missing)
else:
    print(f"PASS: No missing values in training dataset")

# Check feature distributions
feature_cols = ['elo_diff', 'surface_specialization_diff', 'form_diff', 'h2h_overall_advantage']
print(f"\nFeature distributions:")
print(training_df[feature_cols].describe())

In [ ]:
print("=== PERFORMANCE AND CACHING TEST ===")

import time

# Test caching efficiency
test_dates = training_df['match_date'].unique()[:5]
test_players = training_df['player1'].unique()[:10]

print(f"Testing caching with {len(test_dates)} dates and {len(test_players)} players")

# Time first run (cold cache)
start_time = time.time()
for i, date in enumerate(test_dates):
    for j in range(min(5, len(test_players)-1)):  # Test 5 matches per date
        features = feature_engineer.create_match_features(
            date, test_players[j], test_players[j+1], 'Hard'
        )
cold_time = time.time() - start_time

print(f"Cold cache (first run): {cold_time:.2f} seconds")

# Clear some caches and test warm cache
start_time = time.time()
for i, date in enumerate(test_dates):
    for j in range(min(5, len(test_players)-1)):
        features = feature_engineer.create_match_features(
            date, test_players[j], test_players[j+1], 'Hard'
        )
warm_time = time.time() - start_time

print(f"Warm cache (second run): {warm_time:.2f} seconds")
print(f"Speed improvement: {cold_time/warm_time:.1f}x faster")

# Cache size analysis
print(f"\nCache sizes:")
print(f"  ELO cache: {len(feature_engineer.elo_cache)} dates")
print(f"  Surface ELO cache: {len(feature_engineer.surface_elo_cache)} date-surface pairs")
print(f"  Form cache: {len(feature_engineer.form_cache)} date-player combinations")
print(f"  H2H cache: {len(feature_engineer.h2h_cache)} matchup combinations")

In [ ]:
print("=== MODEL-READY FEATURE VALIDATION ===")


# Prepare features for machine learning
feature_columns = [col for col in training_df.columns 
                  if col not in ['match_date', 'player1', 'player2', 'surface', 'tourney_level', 'target']]

X = training_df[feature_columns]
y = training_df['target']

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")
print(f"Feature columns: {feature_columns}")

# Statistical validation
print(f"\n=== FEATURE STATISTICS ===")
print(f"{'Feature':<25} {'Mean':<8} {'Std':<8} {'Min':<8} {'Max':<8}")
print("-" * 60)

for col in feature_columns:
    stats = X[col].describe()
    print(f"{col:<25} {stats['mean']:<8.3f} {stats['std']:<8.3f} {stats['min']:<8.3f} {stats['max']:<8.3f}")

# Correlation analysis
print(f"\n=== FEATURE CORRELATIONS WITH TARGET ===")
correlations = training_df[feature_columns + ['target']].corr()['target'].drop('target')
sorted_corr = correlations.abs().sort_values(ascending=False)

print("Most predictive features (absolute correlation):")
for feature, corr in sorted_corr.head(10).items():
    direction = "+" if correlations[feature] > 0 else "-"
    print(f"  {direction} {feature:<25} {abs(corr):.3f}")

# Check for multicollinearity
print(f"\n=== MULTICOLLINEARITY CHECK ===")
feature_corr_matrix = training_df[feature_columns].corr()
high_corr_pairs = []

for i, col1 in enumerate(feature_columns):
    for j, col2 in enumerate(feature_columns[i+1:], i+1):
        corr_val = abs(feature_corr_matrix.loc[col1, col2])
        if corr_val > 0.8:  # High correlation threshold
            high_corr_pairs.append((col1, col2, corr_val))

if high_corr_pairs:
    print("High correlation pairs (>0.8):")
    for col1, col2, corr in high_corr_pairs:
        print(f"  {col1} <-> {col2}: {corr:.3f}")
else:
    print("PASS: No concerning multicollinearity detected")

print(f"\n Complete pipeline validation successful!")
print(f"Dataset ready for model training: {X.shape[0]} samples, {X.shape[1]} features")